In [52]:
#Khai báo các thư viện/hàm cần dùng
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

In [53]:
#Gọi hàm tiền xử lý ảnh
sys.path.append(str(Path.cwd().parent / 'src' / 'data'))
from preprocess import preprocess_image

In [ ]:
#Đọc dữ liệu huấn luyện
data = pd.read_csv(str(Path.cwd().parent / 'data' / 'train.csv'))

In [ ]:
#Kiểm tra dữ liệu
display(data.head())
print("Kích thước dữ liệu:", data.shape)

,Unnamed: 0,file_name,label
0,0,train_data/a6dcb93f596a43249135678dfcfc17ea.jpg,1
1,1,train_data/041be3153810433ab146bc97d5af505c.jpg,0
2,2,train_data/615df26ce9494e5db2f70e57ce7a3a4f.jpg,1
3,3,train_data/8542fe161d9147be8e835e50c0de39cd.jpg,0
4,4,train_data/5d81fa12bc3b4cea8c94a6700a477cf2.jpg,1


Kích thước dữ liệu: (79950, 3)


In [ ]:
#Loại bỏ dòng thiếu thông tin và tên ảnh trùng lặp
data = data.dropna(subset=['file_name', 'label'])
data = data.drop_duplicates(subset=['file_name']).reset_index(drop=True)

In [ ]:
#Loại bỏ ảnh không tồn tại hoặc không đọc được
image_dir = Path('../data')

def is_valid_image(file_name):
    try:
        with Image.open(image_dir / str(file_name)) as image:
            image.verify()
        return True
    except (OSError, ValueError):
        return False

valid_mask = data['file_name'].apply(is_valid_image)
data = data[valid_mask].reset_index(drop=True)

In [ ]:
#Tách dữ liệu thành tập Train (70%) và tập còn lại (30%)
dt_Train, dt_Temp = train_test_split(data, test_size=0.3, random_state=42, stratify=data['label'])

#Tách tập còn lại thành Validation (15%) và Test (15%)
dt_Val, dt_Test = train_test_split(dt_Temp, test_size=0.5, random_state=42, stratify=dt_Temp['label'])

#Tách tập Train thành dữ liệu ảnh và nhãn
X_train = dt_Train['file_name']
y_train = dt_Train['label']

#Tách tập Validation thành dữ liệu ảnh và nhãn
X_val = dt_Val['file_name']
y_val = dt_Val['label']

#Tách tập Test thành dữ liệu ảnh và nhãn
X_test = dt_Test['file_name']
y_test = dt_Test['label']

In [ ]:
#Augmentation cho tập train
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

In [ ]:
#Resize validation và test , chuẩn hóa pixel về [0, 1]
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])